# Preps

In [11]:
# Grounding
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/zeguru/Projects/AI/tiny-text-transformers


In [12]:
#Imports
from pathlib import Path

import torch
import json
import random

from datetime import datetime
from src.utils.data import split_tokens, get_batch
from src.infra.tokenizer.character import CharacterTokenizer
from src.infra.tokenizer.bpe import BPETokenizer
from src.infra.dataset.answering import QuestionAnsweringDataset
from torch.utils.data import random_split
from src.utils.device import get_device, move_to_device

from src.task.config import ModelConfig,TrainingConfig

from src.task.model import TinyAnsweringModel
from src.task.training.question_answering import train
from src.utils.common import TinyLogger, count_parameters, separator


# Configs

In [13]:
#Model + Training Config

torch.manual_seed(42)

logger = TinyLogger("logs/jupyter_qna_train.log")

logger.log("how tiny?")

# Model hyper params
model_config = ModelConfig(
    d_model=128, 
    number_of_heads=2, 
    context_length=1024
    )

# Training settings
training_config = TrainingConfig(
    num_steps=100,
    eval_interval=10,
    batch_size=128
    )

how tiny?


In [14]:
#Paths
MODEL_DESTINATION = Path("../checkpoints/tiny_squad_answering.pt")

SQUAD_PATH = Path("../data/squad/dev-v1.1.json")

CHAR_TOKENIZER_PATH = Path("../checkpoints/qa_tokenizer.json")
BPE_TOKENIZER_PATH = Path("../checkpoints/bpe_tokenizer.json")

logger.log(f"Squad file: {SQUAD_PATH}")
logger.log(f"Tokenizer file:  {CHAR_TOKENIZER_PATH}")
logger.log(f"Tokenizer file:  {BPE_TOKENIZER_PATH}")

Squad file: ../data/squad/dev-v1.1.json
Tokenizer file:  ../checkpoints/qa_tokenizer.json
Tokenizer file:  ../checkpoints/bpe_tokenizer.json


# Corpus

In [15]:
# Load
with SQUAD_PATH.open("r", encoding="utf-8") as file:
    squad = json.load(file)

# payload
records = []

In [16]:
# Load subset of the corpus

LOAD_ALL_DATA = False  # Change to True when you need everything


records = []

if LOAD_ALL_DATA:
    articles = squad["data"]  # All articles
else:
    articles = [
        article
        for article in squad["data"]
        if article["title"] == "Kenya"
    ]  # Sample: Kenya only

# Load them
for article in articles:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            for answer in qa["answers"]:
                records.append({
                    "question": qa["question"],
                    "context": context,
                    "answer": answer["text"],
                    "answer_start": answer["answer_start"],
                })
                
logger.log(f"Number of records: {len(records)}")

num = random.randint(1, len(records))

logger.log(f"\n=== Sample record: {num} ===")

sample = records[num]
question = sample["question"]
context = sample["context"] 
answer_text = sample["answer"]
answer_start = sample["answer_start"]
answer_end = answer_start + len(answer_text)

logger.log(f"Context: {context}")
logger.log(f"Question: {question}")
logger.log(f"Answer: {answer_text}")
logger.log(f"Answer Start: {answer_start}")
logger.log(f"Answer End: {answer_end}")

Number of records: 691

=== Sample record: 119 ===
Context: At the outbreak of World War I in August 1914, the governors of British East Africa (as the Protectorate was generally known) and German East Africa agreed a truce in an attempt to keep the young colonies out of direct hostilities. Lt Col Paul von Lettow-Vorbeck took command of the German military forces, determined to tie down as many British resources as possible. Completely cut off from Germany, von Lettow conducted an effective guerrilla warfare campaign, living off the land, capturing British supplies, and remaining undefeated. He eventually surrendered in Northern Rhodesia (today Zambia) fourteen days after the Armistice was signed in 1918.
Question: Who was in command of the German military forces?
Answer: Lt Col Paul von Lettow-Vorbeck
Answer Start: 232
Answer End: 262


In [17]:
# Tokenizer Init

char_tokenizer = CharacterTokenizer.load(CHAR_TOKENIZER_PATH)
bpe_tokenizer = BPETokenizer.load(BPE_TOKENIZER_PATH)

#Choose the tokenizer... manual for now
tokenizer = char_tokenizer

vocab_size = tokenizer.vocab_size

logger.log(f"vocabulary size: {vocab_size}: \ntokenizer: {tokenizer}")

vocabulary size: 98: 
tokenizer: <src.infra.tokenizer.character.CharacterTokenizer object at 0x10a87a210>


## Dataset preps

In [18]:
#
dataset = QuestionAnsweringDataset(
    records=records,
    tokenizer=tokenizer,
    max_length=model_config.context_length,         #to manage truncation and padding
    )


n = len(dataset)
train_size = int(0.90 * n)
val_size = int(0.05 * n)
test_size = n - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
    )

logger.log(f"Train examples: {len(train_dataset):,}")
logger.log(f"Val examples:   {len(val_dataset):,}")
logger.log(f"Test examples:  {len(test_dataset):,}")

Train examples: 613
Val examples:   34
Test examples:  35


# Model

In [19]:
# Load the mode

model = TinyAnsweringModel(
    config=model_config,
    vocab_size=tokenizer.vocab_size,
    )

logger.log(model)

Initializing TinyAnsweringModel
	Transformer Block: number of heads: 2
	Transformer Block: number of heads: 2
TinyAnsweringModel(
  (input_representation): InputRepresentation(
    (token_embedding): Embedding(
      (embedding): Embedding(98, 128)
    )
    (positional_embedding): PositionalEmbedding(
      (embedding): Embedding(1024, 128)
    )
  )
  (transformer): Transformer(
    (blocks): ModuleList(
      (0-1): 2 x TransformerBlock(
        (attention): MultiHeadAttention(
          (query): Linear(in_features=128, out_features=128, bias=False)
          (key): Linear(in_features=128, out_features=128, bias=False)
          (value): Linear(in_features=128, out_features=128, bias=False)
          (output_projection): Linear(in_features=128, out_features=128, bias=True)
        )
        (feed_forward): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=128, out_features=256, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(i

# Train

In [20]:
logger.log(f"\tStart: {datetime.now()}===")

model = train(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    training_config=training_config,
    )

logger.log(f"\tEnd: {datetime.now()}===")


	Start: 2026-10-29 18:02:35.780941===
Training examples: 613
Batch size: 128
Batches per epoch: 5
Total batches: 500

Epoch 1/100
	Processing batch : 1 of 5  Device: mps:0
	Processing batch : 2 of 5  Device: mps:0
	Processing batch : 3 of 5  Device: mps:0
	Processing batch : 4 of 5  Device: mps:0
	Processing batch : 5 of 5  Device: mps:0
Batch 5/5 | Time: 6.4899s
==Epoch time: 12.74s
==Average batch time: 2.5350s
	Loss : 12.927733421325684

Epoch 2/100
	Processing batch : 1 of 5  Device: mps:0
	Processing batch : 2 of 5  Device: mps:0
	Processing batch : 3 of 5  Device: mps:0
	Processing batch : 4 of 5  Device: mps:0
	Processing batch : 5 of 5  Device: mps:0
Batch 5/5 | Time: 1.5220s
==Epoch time: 19.70s
==Average batch time: 1.9600s

Epoch 3/100
	Processing batch : 1 of 5  Device: mps:0
	Processing batch : 2 of 5  Device: mps:0
	Processing batch : 3 of 5  Device: mps:0
	Processing batch : 4 of 5  Device: mps:0
	Processing batch : 5 of 5  Device: mps:0
Batch 5/5 | Time: 1.7002s
==Epoch

KeyboardInterrupt: 

# Save the model

In [13]:
logger.log(f"Parameters Count: {count_parameters(model):,}")

torch.save(
    model.state_dict(),
    MODEL_DESTINATION,
)
logger.log(f"Saving model to : {MODEL_DESTINATION}")

Saving model to : ../checkpoints/tiny_answering.pt
